# Soil Moisture Prediction — Phase 1 Baseline Experiments

**Dataset:** 722 images, 6 soil types, 3 kPa bins, 46 series  
**Models:** CNN-Small, ResNet-18, DenseNet-121, EfficientNet-B0, MobileNetV3-Large  
**Hardware:** Kaggle T4 GPU (16 GB VRAM)  
**Preprocessing:** Image tensors (3×224×224, normalized) + MobileNetV3-Small features (576-d)  
**Experiments:** 19 total across 6 categories (see navigation below)

> **Before running:** Attach the dataset `khanraiyan/soil-moisture-detection-dataset`  
> **Accelerator:** Set to **T4 GPU x2**  
> **GPU memory** is flushed after every experiment.  
> **Results** go to `/kaggle/working/results/{exp_name}/`.

## 1. Setup & Dependencies

In [23]:
import subprocess, sys

def _pip(pkg: str):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_pip("xgboost")
_pip("seaborn")
_pip("scipy")
print("Dependencies installed")

Dependencies installed


In [45]:

import random
import gc, json, math, warnings, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision import models, transforms
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
print("All imports loaded")

All imports loaded


## 2. Configuration — Paths & Hyperparameters

> **ALL paths are defined here.** Change BASE_DIR to match your Kaggle dataset.

In [46]:
# ── Paths (edit BASE_DIR if dataset is mounted elsewhere) ──────
BASE_DIR     = Path("/kaggle/input/datasets/khanraiyan/soil-moisture-detection-dataset")
DATA_DIR     = BASE_DIR / "soil-moisture-preprocessed"
FEATURES_DIR = BASE_DIR / "soil-moisture-mobilenet-features"
RESULTS_DIR  = Path("/kaggle/working/results")

# ── Training hyperparameters ───────────────────────────────────
CONFIG = {
    "seed": 42,
    "batch_size": 16,
    "epochs": 100,
    "max_epochs": 100,
    "patience": 15,
    "lr": 1e-3,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "dropout": 0.2,
    "amp": False,
    "num_workers": 2,
    "freeze_backbone": True,
    "unfreeze_layers": 0,
    "scheduler_factor": 0.5,
    "scheduler_patience": 5,
}

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data dir:     {DATA_DIR}")
print(f"Features dir: {FEATURES_DIR}")
print(f"Results dir:  {RESULTS_DIR}")

Data dir:     /kaggle/input/datasets/khanraiyan/soil-moisture-detection-dataset/soil-moisture-preprocessed
Features dir: /kaggle/input/datasets/khanraiyan/soil-moisture-detection-dataset/soil-moisture-mobilenet-features
Results dir:  /kaggle/working/results


## 3. Utility Functions

### 3a. Reproducibility & Device Management

In [26]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def get_device():
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("CUDA not available, using CPU")
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def cleanup_gpu(*args):
    for item in args:
        del item
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

### 3b. Model Builders (5 architectures)

In [27]:
class RegressionHead(nn.Module):
    def __init__(self, in_features, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(256, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

def _configure_model(model, classifier_attr, freeze_backbone, unfreeze_layers, dropout):
    in_features = _extract_backbone_features(model, classifier_attr)
    setattr(model, classifier_attr, RegressionHead(in_features, dropout=dropout))
    if freeze_backbone:
        for name, param in model.named_parameters():
            if not name.startswith(classifier_attr):
                param.requires_grad = False
        if unfreeze_layers > 0:
            modules = [(n, m) for n, m in model.named_children() if n != classifier_attr]
            for _, module in modules[-unfreeze_layers:]:
                for param in module.parameters():
                    param.requires_grad = True
    return model

def _extract_backbone_features(model, classifier_attr):
    classifier = getattr(model, classifier_attr)
    if isinstance(classifier, nn.Linear):
        return classifier.in_features
    if isinstance(classifier, nn.Sequential):
        for layer in classifier:
            if isinstance(layer, nn.Linear):
                return layer.in_features
    with torch.no_grad():
        dummy = torch.zeros(1, 3, 224, 224)
        feat = model.features if hasattr(model, "features") else model
        out = feat(dummy)
        return out.mean([-1, -2]).size(-1) if out.dim() == 4 else out.size(-1)

In [28]:
def build_cnn_small(num_channels=3, num_classes=1):
    backbone = nn.Sequential(
        nn.Conv2d(num_channels, 16, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(16), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d(2),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        nn.Dropout(0.2), nn.Linear(32, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 1),
    )
    class Wrapper(nn.Module):
        def __init__(self, b):
            super().__init__(); self.b = b
        def forward(self, x):
            return self.b(x).squeeze(1)
    return Wrapper(backbone)

def build_resnet18(pretrained=True, freeze_backbone=True, unfreeze_layers=0, dropout=0.2):
    m = models.resnet18(weights="DEFAULT" if pretrained else None)
    return _configure_model(m, "fc", freeze_backbone, unfreeze_layers, dropout)

def build_densenet121(pretrained=True, freeze_backbone=True, unfreeze_layers=0, dropout=0.2):
    m = models.densenet121(weights="DEFAULT" if pretrained else None)
    return _configure_model(m, "classifier", freeze_backbone, unfreeze_layers, dropout)

def build_efficientnet_b0(pretrained=True, freeze_backbone=True, unfreeze_layers=0, dropout=0.2):
    m = models.efficientnet_b0(weights="DEFAULT" if pretrained else None)
    return _configure_model(m, "classifier", freeze_backbone, unfreeze_layers, dropout)

def build_mobilenet_v3_large(pretrained=True, freeze_backbone=True, unfreeze_layers=0, dropout=0.2):
    m = models.mobilenet_v3_large(weights="DEFAULT" if pretrained else None)
    return _configure_model(m, "classifier", freeze_backbone, unfreeze_layers, dropout)

def get_trainable_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable, total

### 3c. Data Loading Functions

In [29]:
def load_metadata(manifest_path, split_path, anomalies_path):
    manifest = pd.read_csv(manifest_path) if Path(manifest_path).exists() else pd.DataFrame()
    split = pd.read_csv(split_path) if Path(split_path).exists() else pd.DataFrame()
    anomalies = set()
    if Path(anomalies_path).exists():
        with open(anomalies_path) as f:
            data = json.load(f)
        for cat in ("contamination",):
            for img_id, info in data.get(cat, {}).items():
                if info.get("action") == "exclude_from_training":
                    anomalies.add(img_id)
    if not anomalies:
        anomalies = {"P0029", "P0166"}
    return manifest, split, anomalies

def create_dataloaders(images, labels, batch_size=16, shuffle=True, num_workers=2):
    t = torch.from_numpy(images).float()
    l = torch.from_numpy(labels).float()
    return DataLoader(TensorDataset(t, l), batch_size=batch_size,
                     shuffle=shuffle, num_workers=num_workers,
                     pin_memory=(torch.cuda.is_available() and num_workers > 0))

### 3d. Training Pipeline

In [47]:
def train_one_epoch(loader, model, criterion, optimizer, device, scaler=None):
    model.train()
    total, n = 0.0, 0
    for batch in loader:
        x = batch[0].to(device, non_blocking=True)
        y = batch[1].to(device, non_blocking=True)
        optimizer.zero_grad()
        if scaler:
            with torch.cuda.amp.autocast():
                loss = criterion(model(x), y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        else:
            loss = criterion(model(x), y)
            loss.backward(); optimizer.step()
        total += loss.item() * x.size(0); n += x.size(0)
    return total / n

@torch.no_grad()
def evaluate(loader, model, device, use_amp=False):
    model.eval()
    total, n = 0.0, 0
    preds, targets = [], []
    for batch in loader:
        x = batch[0].to(device, non_blocking=True)
        y = batch[1].to(device, non_blocking=True)
        if use_amp and device.type == "cuda":
            with torch.cuda.amp.autocast():
                out = model(x)
        else:
            out = model(x)
        total += nn.MSELoss()(out, y).item() * x.size(0)
        n += x.size(0)
        preds.extend(out.cpu().numpy())
        targets.extend(y.cpu().numpy())
    return total / n, np.array(preds), np.array(targets)

In [48]:
def train_model(model, train_loader, val_loader, config, device,
                experiment_name="exp", results_dir=".", clear_cache_after=True):
    exp_dir = Path(results_dir) / experiment_name
    exp_dir.mkdir(parents=True, exist_ok=True)
    lr = config["learning_rate"]
    wd = config.get("weight_decay", 1e-5)
    max_epochs = config.get("max_epochs", 50)
    patience = config.get("patience", 7)
    sched_factor = config.get("scheduler_factor", 0.5)
    sched_patience = config.get("scheduler_patience", 5)

    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=sched_factor, patience=sched_patience)

    trainable, total = get_trainable_params(model)
    print(f"Model: {experiment_name} | Params: {trainable:,} / {total:,}")
    print(f"LR={lr}, WD={wd}, MaxEpochs={max_epochs}, Patience={patience}")

    use_amp = device.type == "cuda" and config.get("amp", False)
    scaler = torch.cuda.amp.GradScaler() if use_amp else None

    model = model.to(device)

    best_val_loss = float("inf")
    best_epoch = -1
    no_improve = 0
    train_losses, val_losses = [], []
    model_path = exp_dir / "best_model.pt"

    for epoch in range(max_epochs):
        tl = train_one_epoch(train_loader, model, criterion, optimizer, device, scaler)
        vl, _, _ = evaluate(val_loader, model, device, use_amp)
        train_losses.append(tl); val_losses.append(vl)
        scheduler.step(vl)
        if vl < best_val_loss:
            best_val_loss = vl; best_epoch = epoch; no_improve = 0
            torch.save(model.state_dict(), model_path)
        else:
            no_improve += 1
        lr_now = optimizer.param_groups[0]["lr"]
        print(f"Epoch {epoch+1:3d}/{max_epochs} | LR: {lr_now:.2e} | Train: {tl:.6f} | Val: {vl:.6f} | Best: {best_val_loss:.6f}")
        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}"); break

    print(f"Best epoch: {best_epoch+1} (Val MSE: {best_val_loss:.6f})")
    result = {"train_losses": train_losses, "val_losses": val_losses,
              "best_epoch": best_epoch, "best_val_loss": best_val_loss,
              "model_path": str(model_path)}
    if clear_cache_after:
        model.cpu(); cleanup_gpu(model, scaler, optimizer)
    return result

### 3e. Metrics & Evaluation

In [49]:
def compute_stratified_metrics(y_true, y_pred, image_ids=None, metadata_df=None):
    y_true, y_pred = np.array(y_true, dtype=float), np.array(y_pred, dtype=float)
    result = {
        "global": {
            "R2": round(r2_score(y_true, y_pred), 4),
            "RMSE": round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 4),
            "MAE": round(float(mean_absolute_error(y_true, y_pred)), 4),
        },
        "per_bin": {}, "per_soil": {}, "rmse_matrix": {},
        "y_true": y_true.tolist(), "y_pred": y_pred.tolist(),
    }
    if metadata_df is None or image_ids is None:
        return result
    meta = metadata_df.set_index("image_id") if "image_id" in metadata_df.columns else metadata_df
    bins, soils = {}, {}
    for i, img_id in enumerate(image_ids):
        if img_id not in meta.index: continue
        row = meta.loc[img_id]
        b = row.get("kpa_bin", "0-10" if y_true[i] <= 10 else "10-20" if y_true[i] <= 20 else "20-22")
        s = row.get("soil_type", "Unknown")
        bins.setdefault(b, {"t": [], "p": []})
        bins[b]["t"].append(y_true[i]); bins[b]["p"].append(y_pred[i])
        soils.setdefault(s, {"t": [], "p": []})
        soils[s]["t"].append(y_true[i]); soils[s]["p"].append(y_pred[i])
    for b, d in bins.items():
        result["per_bin"][b] = {"RMSE": round(float(np.sqrt(mean_squared_error(d["t"], d["p"]))), 4), "count": len(d["t"])}
    for s, d in soils.items():
        result["per_soil"][s] = {"RMSE": round(float(np.sqrt(mean_squared_error(d["t"], d["p"]))), 4), "count": len(d["t"])}
    return result

def save_results(metrics, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f: json.dump(metrics, f, indent=2)

def load_results(path):
    with open(Path(path)) as f: return json.load(f)

In [50]:
def evaluate_model(model, test_loader, device, experiment_name, results_dir,
                   metadata_df=None, image_ids_list=None):
    exp_dir = Path(results_dir) / experiment_name
    exp_dir.mkdir(parents=True, exist_ok=True)
    use_amp = False
    test_loss, test_preds, test_targets = evaluate(test_loader, model, device, use_amp)
    test_rmse = math.sqrt(test_loss)
    print(f"Test MSE: {test_loss:.6f} | RMSE: {test_rmse:.4f}")
    metrics = compute_stratified_metrics(test_targets, test_preds,
        image_ids=image_ids_list, metadata_df=metadata_df)
    g = metrics["global"]
    print(f"Global R2: {g['R2']} | RMSE: {g['RMSE']} | MAE: {g['MAE']}")
    pd.DataFrame({"image_id": image_ids_list or range(len(test_targets)),
                  "kpa_true": test_targets, "kpa_pred": test_preds}
                ).to_csv(exp_dir / "predictions.csv", index=False)
    save_results(metrics, exp_dir / "metrics.json")
    return metrics

### 3f. Plotting Utilities

In [53]:
def plot_training_history(train_losses, val_losses, best_epoch, save_path, exp_name=""):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(train_losses, "o-", label="Train", markersize=3, linewidth=1.5)
    ax.plot(val_losses, "s-", label="Val", markersize=3, linewidth=1.5)
    if best_epoch >= 0:
        ax.axvline(best_epoch, color="green", ls="--", alpha=0.5, label=f"Best ep {best_epoch+1}")
    ax.set_xlabel("Epoch"); ax.set_ylabel("MSE Loss"); ax.set_title(f"Training History -- {exp_name}")
    ax.legend(); ax.grid(alpha=0.3); fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight"); plt.show(); plt.close(fig)

def plot_predictions(y_true, y_pred, save_path, exp_name=""):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(y_true, y_pred, alpha=0.5, s=20)
    lo = min(y_true.min(), y_pred.min()) - 1
    hi = max(y_true.max(), y_pred.max()) + 1
    ax.plot([lo, hi], [lo, hi], "r--", alpha=0.5)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_xlabel("Actual kPa"); ax.set_ylabel("Predicted kPa")
    ax.set_title(f"Predictions -- {exp_name}"); ax.set_aspect("equal"); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(save_path, dpi=150, bbox_inches="tight"); plt.show(); plt.close(fig)

def compare_models(results_dir, model_names):
    rows = []
    for name in model_names:
        p = Path(results_dir) / name / "metrics.json"
        if not p.exists(): continue
        with open(p) as f: m = json.load(f)
        rows.append({"Model": name, **m.get("global", {})})
    return pd.DataFrame(rows).set_index("Model") if rows else pd.DataFrame()

## 4. Load Preprocessed Data

Loads image tensors, metadata, MobileNet features, and creates DataLoaders.
All data is preprocessed — no images on disk needed.

In [36]:
# ── 4a. Image tensors
print("Loading image tensors...")
X_train = np.load(DATA_DIR / "train_images.npy")
y_train = np.load(DATA_DIR / "train_labels.npy")
X_val   = np.load(DATA_DIR / "val_images.npy")
y_val   = np.load(DATA_DIR / "val_labels.npy")
X_test  = np.load(DATA_DIR / "test_images.npy")
y_test  = np.load(DATA_DIR / "test_labels.npy")
train_ids = np.load(DATA_DIR / "train_image_ids.npy", allow_pickle=True)
val_ids   = np.load(DATA_DIR / "val_image_ids.npy", allow_pickle=True)
test_ids  = np.load(DATA_DIR / "test_image_ids.npy", allow_pickle=True)
print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"kPa range: [{min(y_train.min(), y_val.min(), y_test.min()):.1f}, "
      f"{max(y_train.max(), y_val.max(), y_test.max()):.1f}]")

Loading image tensors...
Train: (470, 3, 224, 224) | Val: (119, 3, 224, 224) | Test: (131, 3, 224, 224)
kPa range: [0.0, 21.5]


In [37]:
# ── 4b. Metadata (manifest, split, anomalies)
manifest_df, split_df, anomalies = load_metadata(
    DATA_DIR / "image_manifest.csv",
    DATA_DIR / "split_assignment.csv",
    DATA_DIR / "anomalies.json",
)
# Package as tuple for downstream use
metadata = (manifest_df, split_df, anomalies)
print(f"Soil types: {sorted(manifest_df['soil_type'].unique())}")
print(f"Series:     {manifest_df['series_id'].nunique()}")

Soil types: ['Atel', 'Atel_Doash', 'Bele', 'Bele_Doash', 'Doash', 'Poli']
Series:     46


In [38]:
# ── 4c. MobileNet features
print("Loading MobileNet features...")
features    = np.load(FEATURES_DIR / "mobilenet_features.npy")
feat_labels = np.load(FEATURES_DIR / "mobilenet_labels.npy")
feat_ids    = np.load(FEATURES_DIR / "mobilenet_image_ids.npy", allow_pickle=True)
print(f"Features: {features.shape} | Labels: {feat_labels.shape}")

Loading MobileNet features...
Features: (722, 576) | Labels: (722,)


In [39]:
# ── 4d. PyTorch DataLoaders
train_loader = create_dataloaders(X_train, y_train, CONFIG["batch_size"], shuffle=True)
val_loader   = create_dataloaders(X_val,   y_val,   CONFIG["batch_size"], shuffle=False)
test_loader  = create_dataloaders(X_test,  y_test,  CONFIG["batch_size"], shuffle=False)
print(f"Batches: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}")

Batches: train=30, val=8, test=9


## 5. Exploratory Data Analysis

In [40]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, y, name in zip(axes, [y_train, y_val, y_test], ["Train", "Val", "Test"]):
    ax.hist(y, bins=20, edgecolor="white", color="#2E86AB")
    ax.set_title(f"{name} (n={len(y)})"); ax.set_xlabel("kPa"); ax.set_ylabel("Count")
fig.suptitle("kPa Target Distribution", fontsize=14, y=1.02)
fig.tight_layout(); fig.savefig(RESULTS_DIR / "eda_kpa_distribution.png", dpi=150); plt.show()

In [41]:
fig, ax = plt.subplots(figsize=(8, 4))
cnt = manifest_df["soil_type"].value_counts()
colors = ["#2E86AB", "#A23B72", "#F18F01", "#44BBA4", "#C73E1D", "#3B1F2B"]
ax.bar(cnt.index, cnt.values, color=colors[:len(cnt)], edgecolor="white")
ax.set_title("Images per Soil Type"); ax.set_ylabel("Count")
for i, v in enumerate(cnt.values): ax.text(i, v + 2, str(v), ha="center")
fig.tight_layout(); fig.savefig(RESULTS_DIR / "eda_soil_distribution.png", dpi=150); plt.show()

# EXPERIMENTS

---

## Category A — Individual Model Training

Each model: default CONFIG, AdamW, AMP, early stopping. GPU cleaned after each.

### **A1:** CNN-Small (from scratch)

In [54]:
print("=" * 60)
print(f"  CNN-Small (from scratch)")
print("=" * 60)
exp_dir = RESULTS_DIR / "cnn_small"
exp_dir.mkdir(parents=True, exist_ok=True)
set_seed(CONFIG["seed"])
device = get_device()
model = build_cnn_small(num_channels=3, num_classes=1)
result = train_model(model, train_loader, val_loader, CONFIG, device,
    experiment_name="cnn_small", results_dir=str(RESULTS_DIR), clear_cache_after=False)
metrics = evaluate_model(model, test_loader, device, "cnn_small", str(RESULTS_DIR),
    metadata_df=manifest_df, image_ids_list=test_ids.tolist())
plot_training_history(result["train_losses"], result["val_losses"], result["best_epoch"],
    exp_dir / "training_history.png", "CNN-Small (from scratch)")
plot_predictions(metrics["y_true"], metrics["y_pred"],
    exp_dir / "predictions.png", "CNN-Small (from scratch)")
cleanup_gpu(model, result)
print(f"[OK] CNN-Small (from scratch)")

  CNN-Small (from scratch)
GPU: Tesla T4
Memory: 15.6 GB
Model: cnn_small | Params: 9,537 / 9,537
LR=0.001, WD=0.0001, MaxEpochs=100, Patience=15
Epoch   1/100 | LR: 1.00e-03 | Train: 147.793322 | Val: 300.423673 | Best: 300.423673
Epoch   2/100 | LR: 1.00e-03 | Train: 63.901488 | Val: 140.726932 | Best: 140.726932
Epoch   3/100 | LR: 1.00e-03 | Train: 28.747935 | Val: 62.617158 | Best: 62.617158
Epoch   4/100 | LR: 1.00e-03 | Train: 25.570886 | Val: 50.325822 | Best: 50.325822
Epoch   5/100 | LR: 1.00e-03 | Train: 25.151093 | Val: 70.416006 | Best: 50.325822
Epoch   6/100 | LR: 1.00e-03 | Train: 24.449804 | Val: 64.742754 | Best: 50.325822
Epoch   7/100 | LR: 1.00e-03 | Train: 25.665816 | Val: 55.897153 | Best: 50.325822
Epoch   8/100 | LR: 1.00e-03 | Train: 22.792859 | Val: 48.779989 | Best: 48.779989
Epoch   9/100 | LR: 1.00e-03 | Train: 23.029360 | Val: 65.561138 | Best: 48.779989
Epoch  10/100 | LR: 1.00e-03 | Train: 23.422216 | Val: 62.718815 | Best: 48.779989
Epoch  11/100 | LR:

### **A2:** ResNet-18

In [55]:
print("=" * 60)
print(f"  ResNet-18")
print("=" * 60)
exp_dir = RESULTS_DIR / "resnet18"
exp_dir.mkdir(parents=True, exist_ok=True)
set_seed(CONFIG["seed"])
device = get_device()
model = build_resnet18(pretrained=True, freeze_backbone=True, dropout=CONFIG["dropout"])
result = train_model(model, train_loader, val_loader, CONFIG, device,
    experiment_name="resnet18", results_dir=str(RESULTS_DIR), clear_cache_after=False)
metrics = evaluate_model(model, test_loader, device, "resnet18", str(RESULTS_DIR),
    metadata_df=manifest_df, image_ids_list=test_ids.tolist())
plot_training_history(result["train_losses"], result["val_losses"], result["best_epoch"],
    exp_dir / "training_history.png", "ResNet-18")
plot_predictions(metrics["y_true"], metrics["y_pred"],
    exp_dir / "predictions.png", "ResNet-18")
cleanup_gpu(model, result)
print(f"[OK] ResNet-18")


  ResNet-18
GPU: Tesla T4
Memory: 15.6 GB
Model: resnet18 | Params: 131,585 / 11,308,097
LR=0.001, WD=0.0001, MaxEpochs=100, Patience=15
Epoch   1/100 | LR: 1.00e-03 | Train: 47.388797 | Val: 72.778047 | Best: 72.778047
Epoch   2/100 | LR: 1.00e-03 | Train: 25.948154 | Val: 62.059093 | Best: 62.059093
Epoch   3/100 | LR: 1.00e-03 | Train: 20.727871 | Val: 53.561764 | Best: 53.561764
Epoch   4/100 | LR: 1.00e-03 | Train: 19.358739 | Val: 34.717083 | Best: 34.717083
Epoch   5/100 | LR: 1.00e-03 | Train: 19.100991 | Val: 51.793258 | Best: 34.717083
Epoch   6/100 | LR: 1.00e-03 | Train: 17.672580 | Val: 41.290271 | Best: 34.717083
Epoch   7/100 | LR: 1.00e-03 | Train: 17.406844 | Val: 48.303893 | Best: 34.717083
Epoch   8/100 | LR: 1.00e-03 | Train: 14.010968 | Val: 44.058421 | Best: 34.717083
Epoch   9/100 | LR: 1.00e-03 | Train: 17.316786 | Val: 31.181820 | Best: 31.181820
Epoch  10/100 | LR: 1.00e-03 | Train: 16.262251 | Val: 33.167477 | Best: 31.181820
Epoch  11/100 | LR: 1.00e-03 | Tr

### **A3:** DenseNet-121

In [56]:
print("=" * 60)
print(f"  DenseNet-121")
print("=" * 60)
exp_dir = RESULTS_DIR / "densenet121"
exp_dir.mkdir(parents=True, exist_ok=True)
set_seed(CONFIG["seed"])
device = get_device()
model = build_densenet121(pretrained=True, freeze_backbone=True, dropout=CONFIG["dropout"])
result = train_model(model, train_loader, val_loader, CONFIG, device,
    experiment_name="densenet121", results_dir=str(RESULTS_DIR), clear_cache_after=False)
metrics = evaluate_model(model, test_loader, device, "densenet121", str(RESULTS_DIR),
    metadata_df=manifest_df, image_ids_list=test_ids.tolist())
plot_training_history(result["train_losses"], result["val_losses"], result["best_epoch"],
    exp_dir / "training_history.png", "DenseNet-121")
plot_predictions(metrics["y_true"], metrics["y_pred"],
    exp_dir / "predictions.png", "DenseNet-121")
cleanup_gpu(model, result)
print(f"[OK] DenseNet-121")


  DenseNet-121
GPU: Tesla T4
Memory: 15.6 GB
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 149MB/s] 


Model: densenet121 | Params: 262,657 / 7,216,513
LR=0.001, WD=0.0001, MaxEpochs=100, Patience=15
Epoch   1/100 | LR: 1.00e-03 | Train: 45.932187 | Val: 75.285213 | Best: 75.285213
Epoch   2/100 | LR: 1.00e-03 | Train: 24.622793 | Val: 57.804433 | Best: 57.804433
Epoch   3/100 | LR: 1.00e-03 | Train: 20.973375 | Val: 51.529699 | Best: 51.529699
Epoch   4/100 | LR: 1.00e-03 | Train: 18.241349 | Val: 51.625884 | Best: 51.529699
Epoch   5/100 | LR: 1.00e-03 | Train: 16.562779 | Val: 28.966586 | Best: 28.966586
Epoch   6/100 | LR: 1.00e-03 | Train: 19.423196 | Val: 53.323442 | Best: 28.966586
Epoch   7/100 | LR: 1.00e-03 | Train: 15.883281 | Val: 47.333746 | Best: 28.966586
Epoch   8/100 | LR: 1.00e-03 | Train: 15.342173 | Val: 41.089202 | Best: 28.966586
Epoch   9/100 | LR: 1.00e-03 | Train: 14.663334 | Val: 39.234334 | Best: 28.966586
Epoch  10/100 | LR: 1.00e-03 | Train: 14.748343 | Val: 41.837483 | Best: 28.966586
Epoch  11/100 | LR: 5.00e-04 | Train: 13.516564 | Val: 43.522781 | Best: 

### **A4:** EfficientNet-B0

In [57]:
print("=" * 60)
print(f"  EfficientNet-B0")
print("=" * 60)
exp_dir = RESULTS_DIR / "efficientnet_b0"
exp_dir.mkdir(parents=True, exist_ok=True)
set_seed(CONFIG["seed"])
device = get_device()
model = build_efficientnet_b0(pretrained=True, freeze_backbone=True, dropout=CONFIG["dropout"])
result = train_model(model, train_loader, val_loader, CONFIG, device,
    experiment_name="efficientnet_b0", results_dir=str(RESULTS_DIR), clear_cache_after=False)
metrics = evaluate_model(model, test_loader, device, "efficientnet_b0", str(RESULTS_DIR),
    metadata_df=manifest_df, image_ids_list=test_ids.tolist())
plot_training_history(result["train_losses"], result["val_losses"], result["best_epoch"],
    exp_dir / "training_history.png", "EfficientNet-B0")
plot_predictions(metrics["y_true"], metrics["y_pred"],
    exp_dir / "predictions.png", "EfficientNet-B0")
cleanup_gpu(model, result)
print(f"[OK] EfficientNet-B0")


  EfficientNet-B0
GPU: Tesla T4
Memory: 15.6 GB
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 129MB/s] 


Model: efficientnet_b0 | Params: 328,193 / 4,335,741
LR=0.001, WD=0.0001, MaxEpochs=100, Patience=15
Epoch   1/100 | LR: 1.00e-03 | Train: 72.451700 | Val: 170.472259 | Best: 170.472259
Epoch   2/100 | LR: 1.00e-03 | Train: 37.782752 | Val: 133.977398 | Best: 133.977398
Epoch   3/100 | LR: 1.00e-03 | Train: 32.110981 | Val: 117.441074 | Best: 117.441074
Epoch   4/100 | LR: 1.00e-03 | Train: 25.083763 | Val: 99.337988 | Best: 99.337988
Epoch   5/100 | LR: 1.00e-03 | Train: 23.154966 | Val: 108.722893 | Best: 99.337988
Epoch   6/100 | LR: 1.00e-03 | Train: 18.696770 | Val: 7240.751328 | Best: 99.337988
Epoch   7/100 | LR: 1.00e-03 | Train: 17.174208 | Val: 66.106778 | Best: 66.106778
Epoch   8/100 | LR: 1.00e-03 | Train: 18.119687 | Val: 642.471972 | Best: 66.106778
Epoch   9/100 | LR: 1.00e-03 | Train: 19.142021 | Val: 73.878657 | Best: 66.106778
Epoch  10/100 | LR: 1.00e-03 | Train: 16.409007 | Val: 59177.135988 | Best: 66.106778
Epoch  11/100 | LR: 1.00e-03 | Train: 14.842338 | Val: 1

### **A5:** MobileNetV3-Large

In [58]:
print("=" * 60)
print(f"  MobileNetV3-Large")
print("=" * 60)
exp_dir = RESULTS_DIR / "mobilenet_v3_large"
exp_dir.mkdir(parents=True, exist_ok=True)
set_seed(CONFIG["seed"])
device = get_device()
model = build_mobilenet_v3_large(pretrained=True, freeze_backbone=True, dropout=CONFIG["dropout"])
result = train_model(model, train_loader, val_loader, CONFIG, device,
    experiment_name="mobilenet_v3_large", results_dir=str(RESULTS_DIR), clear_cache_after=False)
metrics = evaluate_model(model, test_loader, device, "mobilenet_v3_large", str(RESULTS_DIR),
    metadata_df=manifest_df, image_ids_list=test_ids.tolist())
plot_training_history(result["train_losses"], result["val_losses"], result["best_epoch"],
    exp_dir / "training_history.png", "MobileNetV3-Large")
plot_predictions(metrics["y_true"], metrics["y_pred"],
    exp_dir / "predictions.png", "MobileNetV3-Large")
cleanup_gpu(model, result)
print(f"[OK] MobileNetV3-Large")


  MobileNetV3-Large
GPU: Tesla T4
Memory: 15.6 GB
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 135MB/s] 


Model: mobilenet_v3_large | Params: 246,273 / 3,218,225
LR=0.001, WD=0.0001, MaxEpochs=100, Patience=15
Epoch   1/100 | LR: 1.00e-03 | Train: 52.766547 | Val: 225.887559 | Best: 225.887559
Epoch   2/100 | LR: 1.00e-03 | Train: 25.174140 | Val: 191.950367 | Best: 191.950367
Epoch   3/100 | LR: 1.00e-03 | Train: 20.881068 | Val: 153.164866 | Best: 153.164866
Epoch   4/100 | LR: 1.00e-03 | Train: 18.969931 | Val: 112.902459 | Best: 112.902459
Epoch   5/100 | LR: 1.00e-03 | Train: 17.144978 | Val: 82.074764 | Best: 82.074764
Epoch   6/100 | LR: 1.00e-03 | Train: 15.282738 | Val: 69.375655 | Best: 69.375655
Epoch   7/100 | LR: 1.00e-03 | Train: 13.450278 | Val: 73.092123 | Best: 69.375655
Epoch   8/100 | LR: 1.00e-03 | Train: 13.257405 | Val: 61.402880 | Best: 61.402880
Epoch   9/100 | LR: 1.00e-03 | Train: 12.903915 | Val: 55.719119 | Best: 55.719119
Epoch  10/100 | LR: 1.00e-03 | Train: 12.605600 | Val: 56.858264 | Best: 55.719119
Epoch  11/100 | LR: 1.00e-03 | Train: 12.165649 | Val: 58.

## Category B — Feature-Based Models

MobileNetV3-Small features (576-d) → RF & XGBoost. No GPU needed.

### **B1:** Random Forest

In [59]:
exp_dir = RESULTS_DIR / "rf_features"; exp_dir.mkdir(parents=True, exist_ok=True)
train_idx = np.isin(feat_ids, train_ids)
test_idx  = np.isin(feat_ids, test_ids)
X_tr_f, y_tr_f = features[train_idx], feat_labels[train_idx]
X_te_f, y_te_f = features[test_idx],  feat_labels[test_idx]
set_seed(CONFIG["seed"])
rf = RandomForestRegressor(n_estimators=300, max_depth=20, n_jobs=-1, random_state=CONFIG["seed"])
rf.fit(X_tr_f, y_tr_f)
y_pred = rf.predict(X_te_f)
metrics = compute_stratified_metrics(y_te_f, y_pred, image_ids=feat_ids[test_idx], metadata_df=manifest_df)
save_results(metrics, exp_dir / "metrics.json")
g = metrics["global"]
print(f"RF: RMSE={g['RMSE']}, R2={g['R2']}, MAE={g['MAE']}")

RF: RMSE=4.7633, R2=-1.1383, MAE=4.1305


### **B2:** XGBoost

In [60]:
exp_dir = RESULTS_DIR / "xgb_features"; exp_dir.mkdir(parents=True, exist_ok=True)
train_idx = np.isin(feat_ids, train_ids)
test_idx  = np.isin(feat_ids, test_ids)
X_tr_f, y_tr_f = features[train_idx], feat_labels[train_idx]
X_te_f, y_te_f = features[test_idx],  feat_labels[test_idx]
set_seed(CONFIG["seed"])
xgb_m = xgb.XGBRegressor(n_estimators=500, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=CONFIG["seed"], n_jobs=-1)
xgb_m.fit(X_tr_f, y_tr_f)
y_pred = xgb_m.predict(X_te_f)
metrics = compute_stratified_metrics(y_te_f, y_pred, image_ids=feat_ids[test_idx], metadata_df=manifest_df)
save_results(metrics, exp_dir / "metrics.json")
g = metrics["global"]
print(f"XGB: RMSE={g['RMSE']}, R2={g['R2']}, MAE={g['MAE']}")

XGB: RMSE=4.8289, R2=-1.1975, MAE=4.1702


## Category C — Ablation Studies

Controlled comparisons using ResNet-18. Each varies one factor at a time.

### **C1:** Input Resolution

In [62]:
def evaluate_model(model, test_loader, device, experiment_name, results_dir,
                   metadata_df=None, image_ids_list=None):
    exp_dir = Path(results_dir) / experiment_name
    exp_dir.mkdir(parents=True, exist_ok=True)
    use_amp = False
    model = model.to(device)
    test_loss, test_preds, test_targets = evaluate(test_loader, model, device, use_amp)
    test_rmse = math.sqrt(test_loss)
    print(f"Test MSE: {test_loss:.6f} | RMSE: {test_rmse:.4f}")
    metrics = compute_stratified_metrics(test_targets, test_preds,
        image_ids=image_ids_list, metadata_df=metadata_df)
    g = metrics["global"]
    print(f"Global R2: {g['R2']} | RMSE: {g['RMSE']} | MAE: {g['MAE']}")
    pd.DataFrame({"image_id": image_ids_list or range(len(test_targets)),
                  "kpa_true": test_targets, "kpa_pred": test_preds}
                ).to_csv(exp_dir / "predictions.csv", index=False)
    save_results(metrics, exp_dir / "metrics.json")
    return metrics
exp_dir = RESULTS_DIR / "resolution_ablation"; exp_dir.mkdir(parents=True, exist_ok=True)
results = []
for res in [224, 112, 56]:
    set_seed(CONFIG["seed"]); device = get_device()
    X_tr = X_train[:,:,:res,:res] if res < 224 else X_train
    X_va = X_val[:,:,:res,:res] if res < 224 else X_val
    X_te = X_test[:,:,:res,:res] if res < 224 else X_test
    tl = create_dataloaders(X_tr, y_train, CONFIG["batch_size"], True)
    vl = create_dataloaders(X_va, y_val, CONFIG["batch_size"], False)
    tel = create_dataloaders(X_te, y_test, CONFIG["batch_size"], False)
    model = build_resnet18(pretrained=True, freeze_backbone=False, dropout=CONFIG["dropout"])
    h = train_model(model, tl, vl, CONFIG, device, f"res_{res}x{res}", str(exp_dir))
    m = evaluate_model(model, tel, device, f"res_{res}x{res}", str(exp_dir))
    results.append({"res": res, **m["global"]}); cleanup_gpu(model, h)
pd.DataFrame(results).to_csv(exp_dir / "results.csv", index=False)
print(results)

GPU: Tesla T4
Memory: 15.6 GB
Model: res_224x224 | Params: 11,308,097 / 11,308,097
LR=0.001, WD=0.0001, MaxEpochs=100, Patience=15
Epoch   1/100 | LR: 1.00e-03 | Train: 37.285994 | Val: 40.897720 | Best: 40.897720
Epoch   2/100 | LR: 1.00e-03 | Train: 16.965089 | Val: 30.222507 | Best: 30.222507
Epoch   3/100 | LR: 1.00e-03 | Train: 16.060239 | Val: 24.281883 | Best: 24.281883
Epoch   4/100 | LR: 1.00e-03 | Train: 13.328223 | Val: 31.753962 | Best: 24.281883
Epoch   5/100 | LR: 1.00e-03 | Train: 12.539743 | Val: 45.839419 | Best: 24.281883
Epoch   6/100 | LR: 1.00e-03 | Train: 13.148159 | Val: 31.061010 | Best: 24.281883
Epoch   7/100 | LR: 1.00e-03 | Train: 12.539480 | Val: 39.472257 | Best: 24.281883
Epoch   8/100 | LR: 1.00e-03 | Train: 9.348609 | Val: 28.724736 | Best: 24.281883
Epoch   9/100 | LR: 5.00e-04 | Train: 10.500809 | Val: 39.667791 | Best: 24.281883
Epoch  10/100 | LR: 5.00e-04 | Train: 9.326603 | Val: 15.415840 | Best: 15.415840
Epoch  11/100 | LR: 5.00e-04 | Train: 7.9

### **C2:** Augmentation

In [63]:
exp_dir = RESULTS_DIR / "augmentation_ablation"; exp_dir.mkdir(parents=True, exist_ok=True)
results = []
for aug, label in [(False, "none"), (True, "light")]:
    set_seed(CONFIG["seed"]); device = get_device()
    if aug:
        t = transforms.Compose([transforms.RandomHorizontalFlip(0.5), transforms.RandomRotation(10)])
        ds = TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(y_train).float())
        class AugDS(TensorDataset):
            def __getitem__(self, i): x, y = super().__getitem__(i); return (t(x) if self.transform else x, y)
            def __init__(self, x, y, t=None): super().__init__(x, y); self.transform = t
        tl = DataLoader(AugDS(torch.from_numpy(X_train).float(), torch.from_numpy(y_train).float(), t),
                        batch_size=CONFIG["batch_size"], shuffle=True)
    else:
        tl = train_loader
    model = build_resnet18(pretrained=True, freeze_backbone=False, dropout=CONFIG["dropout"])
    h = train_model(model, tl, val_loader, CONFIG, device, f"aug_{label}", str(exp_dir))
    m = evaluate_model(model, test_loader, device, f"aug_{label}", str(exp_dir))
    results.append({"aug": label, **m["global"]}); cleanup_gpu(model, h)
pd.DataFrame(results).to_csv(exp_dir / "results.csv", index=False)
print(results)

GPU: Tesla T4
Memory: 15.6 GB
Model: aug_none | Params: 11,308,097 / 11,308,097
LR=0.001, WD=0.0001, MaxEpochs=100, Patience=15
Epoch   1/100 | LR: 1.00e-03 | Train: 37.285994 | Val: 40.897720 | Best: 40.897720
Epoch   2/100 | LR: 1.00e-03 | Train: 16.965089 | Val: 30.222507 | Best: 30.222507
Epoch   3/100 | LR: 1.00e-03 | Train: 16.060239 | Val: 24.281883 | Best: 24.281883
Epoch   4/100 | LR: 1.00e-03 | Train: 13.328223 | Val: 31.753962 | Best: 24.281883
Epoch   5/100 | LR: 1.00e-03 | Train: 12.539743 | Val: 45.839419 | Best: 24.281883
Epoch   6/100 | LR: 1.00e-03 | Train: 13.148159 | Val: 31.061010 | Best: 24.281883
Epoch   7/100 | LR: 1.00e-03 | Train: 12.539480 | Val: 39.472257 | Best: 24.281883
Epoch   8/100 | LR: 1.00e-03 | Train: 9.348609 | Val: 28.724736 | Best: 24.281883
Epoch   9/100 | LR: 5.00e-04 | Train: 10.500809 | Val: 39.667791 | Best: 24.281883
Epoch  10/100 | LR: 5.00e-04 | Train: 9.326603 | Val: 15.415840 | Best: 15.415840
Epoch  11/100 | LR: 5.00e-04 | Train: 7.9129

### **C3:** Normalization

In [64]:
exp_dir = RESULTS_DIR / "normalization_ablation"; exp_dir.mkdir(parents=True, exist_ok=True)
results = []
# Data is pre-normalized with dataset stats. Reverse for "none" comparison.
for rev, label in [(False, "dataset_stats"), (True, "unnormalized")]:
    set_seed(CONFIG["seed"]); device = get_device()
    X_tr = (X_train * 0.225 + 0.485) if rev else X_train
    X_va = (X_val * 0.225 + 0.485) if rev else X_val
    X_te = (X_test * 0.225 + 0.485) if rev else X_test
    tl = create_dataloaders(X_tr, y_train, CONFIG["batch_size"], True)
    vl = create_dataloaders(X_va, y_val, CONFIG["batch_size"], False)
    tel = create_dataloaders(X_te, y_test, CONFIG["batch_size"], False)
    model = build_resnet18(pretrained=True, freeze_backbone=False, dropout=CONFIG["dropout"])
    h = train_model(model, tl, vl, CONFIG, device, f"norm_{label}", str(exp_dir))
    m = evaluate_model(model, tel, device, f"norm_{label}", str(exp_dir))
    results.append({"norm": label, **m["global"]}); cleanup_gpu(model, h)
pd.DataFrame(results).to_csv(exp_dir / "results.csv", index=False)
print(results)

GPU: Tesla T4
Memory: 15.6 GB
Model: norm_dataset_stats | Params: 11,308,097 / 11,308,097
LR=0.001, WD=0.0001, MaxEpochs=100, Patience=15
Epoch   1/100 | LR: 1.00e-03 | Train: 37.285994 | Val: 40.897720 | Best: 40.897720
Epoch   2/100 | LR: 1.00e-03 | Train: 16.965089 | Val: 30.222507 | Best: 30.222507
Epoch   3/100 | LR: 1.00e-03 | Train: 16.060239 | Val: 24.281883 | Best: 24.281883
Epoch   4/100 | LR: 1.00e-03 | Train: 13.328223 | Val: 31.753962 | Best: 24.281883
Epoch   5/100 | LR: 1.00e-03 | Train: 12.539743 | Val: 45.839419 | Best: 24.281883
Epoch   6/100 | LR: 1.00e-03 | Train: 13.148159 | Val: 31.061010 | Best: 24.281883
Epoch   7/100 | LR: 1.00e-03 | Train: 12.539480 | Val: 39.472257 | Best: 24.281883
Epoch   8/100 | LR: 1.00e-03 | Train: 9.348609 | Val: 28.724736 | Best: 24.281883
Epoch   9/100 | LR: 5.00e-04 | Train: 10.500809 | Val: 39.667791 | Best: 24.281883
Epoch  10/100 | LR: 5.00e-04 | Train: 9.326603 | Val: 15.415840 | Best: 15.415840
Epoch  11/100 | LR: 5.00e-04 | Tra

### **C4:** Freezing Strategy

In [65]:
exp_dir = RESULTS_DIR / "freezing_ablation"; exp_dir.mkdir(parents=True, exist_ok=True)
results = []
for freeze, unfreeze, label in [(True, 0, "frozen"), (True, 2, "unfreeze_2"), (False, 0, "full_ft")]:
    set_seed(CONFIG["seed"]); device = get_device()
    model = build_resnet18(pretrained=True, freeze_backbone=freeze, unfreeze_layers=unfreeze, dropout=CONFIG["dropout"])
    h = train_model(model, train_loader, val_loader, CONFIG, device, f"freeze_{label}", str(exp_dir))
    m = evaluate_model(model, test_loader, device, f"freeze_{label}", str(exp_dir))
    results.append({"freeze": label, **m["global"]}); cleanup_gpu(model, h)
pd.DataFrame(results).to_csv(exp_dir / "results.csv", index=False)
print(results)

GPU: Tesla T4
Memory: 15.6 GB
Model: freeze_frozen | Params: 131,585 / 11,308,097
LR=0.001, WD=0.0001, MaxEpochs=100, Patience=15
Epoch   1/100 | LR: 1.00e-03 | Train: 47.388797 | Val: 72.778047 | Best: 72.778047
Epoch   2/100 | LR: 1.00e-03 | Train: 25.948154 | Val: 62.059093 | Best: 62.059093
Epoch   3/100 | LR: 1.00e-03 | Train: 20.727871 | Val: 53.561764 | Best: 53.561764
Epoch   4/100 | LR: 1.00e-03 | Train: 19.358739 | Val: 34.717083 | Best: 34.717083
Epoch   5/100 | LR: 1.00e-03 | Train: 19.100991 | Val: 51.793258 | Best: 34.717083
Epoch   6/100 | LR: 1.00e-03 | Train: 17.672580 | Val: 41.290271 | Best: 34.717083
Epoch   7/100 | LR: 1.00e-03 | Train: 17.406844 | Val: 48.303893 | Best: 34.717083
Epoch   8/100 | LR: 1.00e-03 | Train: 14.010968 | Val: 44.058421 | Best: 34.717083
Epoch   9/100 | LR: 1.00e-03 | Train: 17.316786 | Val: 31.181820 | Best: 31.181820
Epoch  10/100 | LR: 1.00e-03 | Train: 16.262251 | Val: 33.167477 | Best: 31.181820
Epoch  11/100 | LR: 1.00e-03 | Train: 15

### **C5:** Per-Soil (RF)

In [66]:
exp_dir = RESULTS_DIR / "per_soil_ablation"; exp_dir.mkdir(parents=True, exist_ok=True)
results = []
for soil in sorted(manifest_df["soil_type"].unique()):
    soil_ids = manifest_df[manifest_df["soil_type"]==soil]["image_id"].tolist()
    tr_s = [i for i in soil_ids if i in train_ids]
    te_s = [i for i in soil_ids if i in test_ids]
    if len(tr_s) < 5 or len(te_s) < 3: continue
    tr_m = np.isin(feat_ids, tr_s); te_m = np.isin(feat_ids, te_s)
    rf = RandomForestRegressor(n_estimators=200, max_depth=15, n_jobs=-1, random_state=CONFIG["seed"])
    rf.fit(features[tr_m], feat_labels[tr_m])
    p = rf.predict(features[te_m])
    r = {"soil": soil, "n_train": sum(tr_m), "n_test": sum(te_m),
         "RMSE": round(float(np.sqrt(mean_squared_error(feat_labels[te_m], p))), 4),
         "R2": round(float(r2_score(feat_labels[te_m], p)), 4)}
    results.append(r); print(f"  {soil}: RMSE={r['RMSE']}")
pd.DataFrame(results).to_csv(exp_dir / "results.csv", index=False)

  Atel: RMSE=1.8387
  Bele: RMSE=7.2012
  Bele_Doash: RMSE=2.7909
  Poli: RMSE=6.4154


## Category D — Cross-Validation

Leave-One-Series-Out: 46 folds, one per series. Evaluates generalization to unseen series.

### **D1:** LOSO Cross-Validation

In [69]:
exp_dir = RESULTS_DIR / "loso_cv"; exp_dir.mkdir(parents=True, exist_ok=True)
series_list = sorted(manifest_df["series_id"].unique())
print(f"Total series: {len(series_list)}")
fold_results = []
for series_id in series_list:
    te_ids_s = manifest_df[manifest_df["series_id"]==series_id]["image_id"].tolist()
    tr_s = [i for i in train_ids if i not in te_ids_s]
    if len(tr_s) < 10 or len(te_ids_s) < 2: continue
    tr_i = [list(train_ids).index(i) for i in tr_s if i in train_ids]
    te_i = [list(test_ids).index(i) for i in te_ids_s if i in test_ids]
    if len(te_i) < 2: continue
    X_tr = np.concatenate([X_train[tr_i], X_val[[list(val_ids).index(i) for i in tr_s if i in val_ids]]], 0) if any(i in val_ids for i in tr_s) else X_train[tr_i]
    y_tr = np.concatenate([y_train[tr_i], y_val[[list(val_ids).index(i) for i in tr_s if i in val_ids]]], 0) if any(i in val_ids for i in tr_s) else y_train[tr_i]
    tl = create_dataloaders(X_tr, y_tr, CONFIG["batch_size"], True)
    tel = create_dataloaders(X_test[te_i], y_test[te_i], CONFIG["batch_size"], False)
    set_seed(CONFIG["seed"]); device = get_device()
    model = build_resnet18(pretrained=True, freeze_backbone=False, dropout=CONFIG["dropout"])
    h = train_model(model, tl, tel, CONFIG, device, f"loso_{series_id}", str(exp_dir), clear_cache_after=False)
    _, p, t = evaluate(tel, model, device, False)
    rmse = float(np.sqrt(mean_squared_error(t, p)))
    fold_results.append({"series": series_id, "rmse": rmse, "n_test": len(te_i)})
    print(f"  Series {series_id}: RMSE={rmse:.4f}"); cleanup_gpu(model, h)
df = pd.DataFrame(fold_results)
df.to_csv(exp_dir / "loso_results.csv", index=False)
print(f"Mean RMSE: {df['rmse'].mean():.4f} +/- {df['rmse'].std():.4f}")

Total series: 46
GPU: Tesla T4
Memory: 15.6 GB
Model: loso_S0003 | Params: 11,308,097 / 11,308,097
LR=0.001, WD=0.0001, MaxEpochs=100, Patience=15
Epoch   1/100 | LR: 1.00e-03 | Train: 37.285994 | Val: 186.173752 | Best: 186.173752
Epoch   2/100 | LR: 1.00e-03 | Train: 16.965089 | Val: 0.071512 | Best: 0.071512
Epoch   3/100 | LR: 1.00e-03 | Train: 16.060239 | Val: 0.000024 | Best: 0.000024
Epoch   4/100 | LR: 1.00e-03 | Train: 13.328223 | Val: 2.097986 | Best: 0.000024
Epoch   5/100 | LR: 1.00e-03 | Train: 12.539743 | Val: 0.157573 | Best: 0.000024
Epoch   6/100 | LR: 1.00e-03 | Train: 13.148159 | Val: 0.561478 | Best: 0.000024
Epoch   7/100 | LR: 1.00e-03 | Train: 12.539480 | Val: 0.128896 | Best: 0.000024
Epoch   8/100 | LR: 1.00e-03 | Train: 9.348609 | Val: 0.337003 | Best: 0.000024
Epoch   9/100 | LR: 5.00e-04 | Train: 10.500809 | Val: 3.344981 | Best: 0.000024
Epoch  10/100 | LR: 5.00e-04 | Train: 9.326603 | Val: 2.830405 | Best: 0.000024
Epoch  11/100 | LR: 5.00e-04 | Train: 7.9

## Category E — Statistical Validation

Bootstrap confidence intervals + anomaly sensitivity analysis.

### **E1:** Bootstrap CI (1,000 iterations)

In [71]:
exp_dir = RESULTS_DIR / "bootstrap_ci"; exp_dir.mkdir(parents=True, exist_ok=True)
set_seed(CONFIG["seed"]); device = get_device()
model = build_resnet18(pretrained=True, freeze_backbone=False, dropout=CONFIG["dropout"])
h = train_model(model, train_loader, val_loader, CONFIG, device, "bootstrap_ref", str(exp_dir), clear_cache_after=False)
_, preds, true_vals = evaluate(test_loader, model, device, False)
n = len(true_vals); r2s, rmses = [], []
for i in range(1000):
    idx = np.random.choice(n, n, replace=True)
    r2s.append(r2_score(true_vals[idx], preds[idx]))
    rmses.append(float(np.sqrt(mean_squared_error(true_vals[idx], preds[idx]))))
r2_ci = [np.percentile(r2s, 2.5), np.percentile(r2s, 97.5)]
rm_ci = [np.percentile(rmses, 2.5), np.percentile(rmses, 97.5)]
print(f"R2 median={np.median(r2s):.4f} 95%CI=[{r2_ci[0]:.4f},{r2_ci[1]:.4f}]")
print(f"RMSE median={np.median(rmses):.4f} 95%CI=[{rm_ci[0]:.4f},{rm_ci[1]:.4f}]")
save_results({"r2_ci": r2_ci, "rmse_ci": rm_ci}, exp_dir / "bootstrap_ci.json")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(r2s, bins=50, color="#2E86AB", edgecolor="white")
axes[0].axvline(r2_ci[0], color="red", ls="--"); axes[0].axvline(r2_ci[1], color="red", ls="--")
axes[0].set_title("Bootstrap R2")
axes[1].hist(rmses, bins=50, color="#A23B72", edgecolor="white")
axes[1].axvline(rm_ci[0], color="red", ls="--"); axes[1].axvline(rm_ci[1], color="red", ls="--")
axes[1].set_title("Bootstrap RMSE")
fig.tight_layout(); fig.savefig(exp_dir / "bootstrap.png", dpi=150); plt.show()
cleanup_gpu(model, h)

GPU: Tesla T4
Memory: 15.6 GB
Model: bootstrap_ref | Params: 11,308,097 / 11,308,097
LR=0.001, WD=0.0001, MaxEpochs=100, Patience=15
Epoch   1/100 | LR: 1.00e-03 | Train: 37.285994 | Val: 40.897720 | Best: 40.897720
Epoch   2/100 | LR: 1.00e-03 | Train: 16.965089 | Val: 30.222507 | Best: 30.222507
Epoch   3/100 | LR: 1.00e-03 | Train: 16.060239 | Val: 24.281883 | Best: 24.281883
Epoch   4/100 | LR: 1.00e-03 | Train: 13.328223 | Val: 31.753962 | Best: 24.281883
Epoch   5/100 | LR: 1.00e-03 | Train: 12.539743 | Val: 45.839419 | Best: 24.281883
Epoch   6/100 | LR: 1.00e-03 | Train: 13.148159 | Val: 31.061010 | Best: 24.281883
Epoch   7/100 | LR: 1.00e-03 | Train: 12.539480 | Val: 39.472257 | Best: 24.281883
Epoch   8/100 | LR: 1.00e-03 | Train: 9.348609 | Val: 28.724736 | Best: 24.281883
Epoch   9/100 | LR: 5.00e-04 | Train: 10.500809 | Val: 39.667791 | Best: 24.281883
Epoch  10/100 | LR: 5.00e-04 | Train: 9.326603 | Val: 15.415840 | Best: 15.415840
Epoch  11/100 | LR: 5.00e-04 | Train: 7

### **E2:** Anomaly Sensitivity

In [72]:
exp_dir = RESULTS_DIR / "anomaly_sensitivity"; exp_dir.mkdir(parents=True, exist_ok=True)
results = []
for exclude, label in [(True, "excluded"), (False, "included")]:
    set_seed(CONFIG["seed"]); device = get_device()
    if exclude:
        mask = ~np.isin(train_ids, list(anomalies))
        tl = create_dataloaders(X_train[mask], y_train[mask], CONFIG["batch_size"], True)
    else:
        tl = train_loader
    model = build_resnet18(pretrained=True, freeze_backbone=False, dropout=CONFIG["dropout"])
    h = train_model(model, tl, val_loader, CONFIG, device, f"anomaly_{label}", str(exp_dir), clear_cache_after=False)
    m = evaluate_model(model, test_loader, device, f"anomaly_{label}", str(exp_dir))
    results.append({"cond": label, **m["global"]}); cleanup_gpu(model, h)
pd.DataFrame(results).to_csv(exp_dir / "results.csv", index=False)
print(results)

GPU: Tesla T4
Memory: 15.6 GB
Model: anomaly_excluded | Params: 11,308,097 / 11,308,097
LR=0.001, WD=0.0001, MaxEpochs=100, Patience=15
Epoch   1/100 | LR: 1.00e-03 | Train: 37.285994 | Val: 40.897720 | Best: 40.897720
Epoch   2/100 | LR: 1.00e-03 | Train: 16.965089 | Val: 30.222507 | Best: 30.222507
Epoch   3/100 | LR: 1.00e-03 | Train: 16.060239 | Val: 24.281883 | Best: 24.281883
Epoch   4/100 | LR: 1.00e-03 | Train: 13.328223 | Val: 31.753962 | Best: 24.281883
Epoch   5/100 | LR: 1.00e-03 | Train: 12.539743 | Val: 45.839419 | Best: 24.281883
Epoch   6/100 | LR: 1.00e-03 | Train: 13.148159 | Val: 31.061010 | Best: 24.281883
Epoch   7/100 | LR: 1.00e-03 | Train: 12.539480 | Val: 39.472257 | Best: 24.281883
Epoch   8/100 | LR: 1.00e-03 | Train: 9.348609 | Val: 28.724736 | Best: 24.281883
Epoch   9/100 | LR: 5.00e-04 | Train: 10.500809 | Val: 39.667791 | Best: 24.281883
Epoch  10/100 | LR: 5.00e-04 | Train: 9.326603 | Val: 15.415840 | Best: 15.415840
Epoch  11/100 | LR: 5.00e-04 | Train

## Category F — Ensemble & Analysis

Model averaging, per-crop evaluation, error analysis, feature importance.

### **F1:** Ensemble (Average of 5)

In [73]:
exp_dir = RESULTS_DIR / "ensemble"; exp_dir.mkdir(parents=True, exist_ok=True)
model_names = ["cnn_small", "resnet18", "densenet121", "efficientnet_b0", "mobilenet_v3_large"]
preds_list = []
for mn in model_names:
    p = RESULTS_DIR / mn / "metrics.json"
    if p.exists():
        m = load_results(p); preds_list.append(m["y_pred"]); print(f"Loaded {mn}")
if len(preds_list) >= 2:
    ens = np.mean(preds_list, axis=0)
    ref = load_results(RESULTS_DIR / model_names[0] / "metrics.json")
    m = compute_stratified_metrics(ref["y_true"], ens)
    save_results(m, exp_dir / "metrics.json"); g = m["global"]
    print(f"Ensemble: RMSE={g['RMSE']}, R2={g['R2']}")

Loaded cnn_small
Loaded resnet18
Loaded densenet121
Loaded efficientnet_b0
Loaded mobilenet_v3_large
Ensemble: RMSE=5.0766, R2=-1.4288


### **F2:** Per-Crop Analysis

In [74]:
exp_dir = RESULTS_DIR / "per_crop_analysis"; exp_dir.mkdir(parents=True, exist_ok=True)
crop_map = {"P": "Potato", "M": "Maize", "W": "Wheat", "S": "Soybean",
            "C": "Cotton", "R": "Rice", "B": "Barley", "O": "Oat"}
manifest_df["crop"] = manifest_df["series_id"].str[0].map(crop_map).fillna("Other")
ref = load_results(RESULTS_DIR / "resnet18" / "metrics.json") if (RESULTS_DIR / "resnet18" / "metrics.json").exists() else None
if ref:
    yt, yp = np.array(ref["y_true"]), np.array(ref["y_pred"])
    results = []
    for crop in sorted(manifest_df["crop"].unique()):
        mask = np.isin(test_ids, manifest_df[manifest_df["crop"]==crop]["image_id"])
        if mask.sum() < 3: continue
        results.append({"crop": crop, "n": int(mask.sum()),
            "RMSE": round(float(np.sqrt(mean_squared_error(yt[mask], yp[mask]))), 4),
            "R2": round(float(r2_score(yt[mask], yp[mask])), 4)})
    pd.DataFrame(results).to_csv(exp_dir / "results.csv", index=False)
    print(pd.DataFrame(results))

      crop    n    RMSE      R2
0  Soybean  131  5.2145 -1.5626


### **F3:** Error Analysis

In [75]:
exp_dir = RESULTS_DIR / "error_analysis"; exp_dir.mkdir(parents=True, exist_ok=True)
ref = load_results(RESULTS_DIR / "resnet18" / "metrics.json") if (RESULTS_DIR / "resnet18" / "metrics.json").exists() else None
if ref:
    yt, yp = np.array(ref["y_true"]), np.array(ref["y_pred"])
    errs = yp - yt; abs_e = np.abs(errs)
    print(f"Mean error: {errs.mean():.4f}, Std: {errs.std():.4f}, Max abs: {abs_e.max():.4f}")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(errs, bins=30, color="#A23B72", edgecolor="white")
    axes[0].axvline(0, color="red", ls="--"); axes[0].set_title("Error Distribution")
    axes[1].scatter(yt, errs, alpha=0.6, color="#2E86AB")
    axes[1].axhline(0, color="red", ls="--"); axes[1].set_xlabel("Actual kPa"); axes[1].set_ylabel("Residual")
    axes[1].set_title("Residuals vs Actual")
    fig.tight_layout(); fig.savefig(exp_dir / "error_analysis.png", dpi=150); plt.show()
    bin_results = []
    for lo, hi, name in [(0, 10, "0-10"), (10, 20, "10-20"), (20, 22, "20-22")]:
        m = (yt >= lo) & (yt < hi)
        if m.sum() > 0:
            bin_results.append({"bin": name, "n": int(m.sum()),
                "RMSE": round(float(np.sqrt(mean_squared_error(yt[m], yp[m]))), 4)})
    pd.DataFrame(bin_results).to_csv(exp_dir / "per_bin_errors.csv", index=False)
    print(pd.DataFrame(bin_results))

Mean error: -4.3964, Std: 2.8040, Max abs: 10.1054
     bin   n    RMSE
0   0-10   2  7.2638
1  10-20  92  5.0447
2  20-22  37  5.4900


### **F4:** Feature Importance (Permutation)

In [76]:
exp_dir = RESULTS_DIR / "feature_importance"; exp_dir.mkdir(parents=True, exist_ok=True)
train_idx = np.isin(feat_ids, train_ids)
test_idx  = np.isin(feat_ids, test_ids)
rf = RandomForestRegressor(n_estimators=200, max_depth=15, n_jobs=-1, random_state=CONFIG["seed"])
rf.fit(features[train_idx], feat_labels[train_idx])
result = permutation_importance(rf, features[test_idx], feat_labels[test_idx],
    n_repeats=10, random_state=CONFIG["seed"], n_jobs=-1)
top_n = 20; idx = result.importances_mean.argsort()[::-1][:top_n]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(top_n), result.importances_mean[idx][::-1], xerr=result.importances_std[idx][::-1],
        color="#2E86AB", edgecolor="white")
ax.set_yticks(range(top_n)); ax.set_yticklabels([f"F{i}" for i in idx[::-1]])
ax.set_xlabel("Permutation Importance"); ax.set_title("Top 20 Features (RF)")
fig.tight_layout(); fig.savefig(exp_dir / "feature_importance.png", dpi=150); plt.show()
pd.DataFrame({"feature": idx, "importance": result.importances_mean[idx]}
           ).to_csv(exp_dir / "top20.csv", index=False)

# SUMMARY

In [77]:
print("=" * 60)
print("  MODEL COMPARISON")
print("=" * 60)
model_names = ["cnn_small", "resnet18", "densenet121", "efficientnet_b0",
              "mobilenet_v3_large", "rf_features", "xgb_features"]
df = compare_models(RESULTS_DIR, model_names + ["ensemble"])
if not df.empty:
    df = df.sort_values("RMSE")
    print(df.to_string(float_format=lambda x: f"{x:.4f}"))
    df.to_csv(RESULTS_DIR / "model_comparison.csv")
    best = df["RMSE"].idxmin()
    print(f"\nBest model: {best} (RMSE={df.loc[best, 'RMSE']:.4f})")

  MODEL COMPARISON
                        R2   RMSE    MAE
Model                                   
rf_features        -1.1383 4.7633 4.1305
xgb_features       -1.1975 4.8289 4.1702
ensemble           -1.4288 5.0766 4.6699
cnn_small          -1.5090 5.1597 4.7268
resnet18           -1.5626 5.2145 4.6990
efficientnet_b0    -1.8221 5.4723 4.7368
mobilenet_v3_large -1.8448 5.4942 4.8823
densenet121        -1.9672 5.6112 4.9595

Best model: rf_features (RMSE=4.7633)


In [78]:
zip_path = RESULTS_DIR.parent / "phase1_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fpath in RESULTS_DIR.rglob("*"):
        if fpath.is_file():
            zf.write(fpath, arcname=fpath.relative_to(RESULTS_DIR))
print(f"Results: {zip_path} ({zip_path.stat().st_size/1024/1024:.1f} MB)")
print("DONE - Navigate to /kaggle/working/ to download")

Results: /kaggle/working/phase1_results.zip (815.5 MB)
DONE - Navigate to /kaggle/working/ to download
